# 🧹 Preprocesamiento de Datos — Proyecto RPA

**Fuente de datos:** `data/database/Procesos.db` (SQLite)  
**Salida:** `data/database/Procesos_clean.db` (tablas limpias)

> **Decisión de arquitectura:** Se lee directamente desde `Procesos.db` sin copiarlo.
> Los resultados limpios se escriben en `Procesos_clean.db` para mantener el dato
> original intacto y garantizar un pipeline reproducible y no destructivo.

| Fase | Descripción |
|------|-------------|
| 1 | Validación de entorno y rutas |
| 2 | Carga de librerías |
| 3 | Lectura desde SQLite |
| 4 | Auditoría de calidad |
| 5 | Limpieza por tabla |
| 6 | Normalización numérica |
| 7 | Exportación a Procesos_clean.db |

## 1 · Validación de Entorno

Se verifica que `Procesos.db` exista y que el directorio tenga permisos de escritura
para generar `Procesos_clean.db`. Fallar rápido evita ejecutar transformaciones
costosas sobre datos inaccesibles.

In [5]:
import os, sys

NOTEBOOK_DIR  = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT  = os.path.dirname(NOTEBOOK_DIR)
DB_DIR        = os.path.join(PROJECT_ROOT, 'data', 'database')

DB_SOURCE = os.path.join(DB_DIR, 'Procesos.db')
DB_OUTPUT = os.path.join(DB_DIR, 'Procesos_clean.db')

# Validar fuente
if not os.path.isfile(DB_SOURCE):
    print(f'[ERROR] No se encontró: {DB_SOURCE}')
    sys.exit(1)
else:
    size_mb = os.path.getsize(DB_SOURCE) / 1_048_576
    print(f'  ✓ Procesos.db encontrado  ({size_mb:.1f} MB)')

# Validar permisos de escritura en DB_DIR
if os.access(DB_DIR, os.W_OK):
    print(f'  ✓ Permisos de escritura OK: {DB_DIR}')
else:
    print(f'  ✗ Sin permisos de escritura en: {DB_DIR}')
    sys.exit(1)

  ✓ Procesos.db encontrado  (68.4 MB)
  ✓ Permisos de escritura OK: c:\Users\esneydergm\OneDrive - Caja de Compensacion Familiar de Antioquia COMFAMA\Escritorio\Proyecto1Especializacion\.claude\worktrees\hungry-ritchie-69d57f\data\database


## 2 · Carga de Librerías

Cada `import` está envuelto en un `try/except` para producir mensajes de error
accionables en lugar de trazas crípticas.

In [7]:
import sqlite3

try:
    import pandas as pd
    print(f'  pandas  {pd.__version__}  ✓')
except ImportError:
    raise ImportError('pandas no instalado → ejecuta: uv add pandas')

try:
    import numpy as np
    print(f'  numpy   {np.__version__}  ✓')
except ImportError:
    raise ImportError('numpy no instalado → ejecuta: uv add numpy')

try:
    from sklearn.preprocessing import MinMaxScaler
    print('  scikit-learn          ✓')
except ImportError:
    raise ImportError('scikit-learn no instalado → ejecuta: uv add scikit-learn')

ImportError: pandas no instalado → ejecuta: uv add pandas

## 3 · Lectura desde SQLite

`pd.read_sql()` ejecuta una consulta SQL directamente sobre `Procesos.db` y
retorna un DataFrame. Se abre la conexión en modo **solo lectura** (`uri=True`
con `mode=ro`) para garantizar que nunca se modifique la fuente original.

In [ ]:
URI_RO = f'file:{DB_SOURCE}?mode=ro'

def leer_tabla(tabla: str, conn) -> pd.DataFrame:
    df = pd.read_sql(f'SELECT * FROM "{tabla}"', conn)
    print(f'  {tabla}: {df.shape[0]:,} filas × {df.shape[1]} columnas')
    return df

with sqlite3.connect(URI_RO, uri=True) as conn_ro:
    # Listar tablas disponibles
    tablas = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        conn_ro
    )['name'].tolist()
    print(f'Tablas en Procesos.db: {tablas}\n')

    df_tiempos   = leer_tabla('TiemposManuales', conn_ro)
    df_roles     = leer_tabla('RolesAreas',      conn_ro)
    df_registros = leer_tabla('RegistrosDPA',    conn_ro)

## 4 · Auditoría de Calidad

Antes de modificar cualquier dato se documenta el estado inicial: tipos, nulos
y cardinalidad. Esto permite elegir la estrategia de tratamiento correcta.

In [ ]:
def auditoria(df: pd.DataFrame, nombre: str) -> None:
    n = len(df)
    nulos = df.isnull().sum()
    # SQLite guarda NULL literal como string 'NULL' en algunos casos
    nulos_str = (df == 'NULL').sum()
    resumen = pd.DataFrame({
        'dtype':          df.dtypes,
        'nulos_real':     nulos,
        'nulos_str_NULL': nulos_str,
        'pct_nulos':      (nulos / n * 100).round(1),
        'unique':         df.nunique()
    })
    print(f'\n{"="*58}')
    print(f' {nombre}  ({n:,} registros)')
    print(f'{"="*58}')
    print(resumen.to_string())

auditoria(df_tiempos,   'TiemposManuales')
auditoria(df_roles,     'RolesAreas')
auditoria(df_registros, 'RegistrosDPA')

## 5 · Limpieza — TiemposManuales

| Columna | Problema | Estrategia |
|---------|----------|------------|
| `Proyecto` | 100 % `'NULL'` literal (herencia del CSV original) | Eliminar columna |
| `TiempoManualHoras` | Vacíos y nulos | Conservar como **0** |
| `FechaSalidaProduccion` | ~30 % vacíos | Convertir a `datetime`, conservar `NaT` |
| `Desarrollador` | ~20 % vacíos | Imputar `'Sin asignar'` |
| `Tecnologia` | Capitalización inconsistente (`Uipath`/`UiPath`) | Mapa de estandarización |

> **¿Por qué conservar 0 en `TiempoManualHoras`?** Esta columna es determinante
> para calcular el total de dinero ahorrado. Imputar con la mediana introduciría
> sesgo en ese cálculo: un bot sin tiempo registrado aporta **0** al ahorro,
> no el valor promedio del resto del portafolio.

In [ ]:
tm = df_tiempos.copy()

# 1. Normalizar 'NULL' literal → NaN y eliminar columna si es 100 % nula
tm = tm.replace('NULL', np.nan)
if tm['Proyecto'].isna().all():
    tm = tm.drop(columns=['Proyecto'])
    print('  [OK] Columna Proyecto eliminada (100 % NULL)')

# 2. TiempoManualHoras → numérico; vacíos y nulos → 0
#    Conservar 0 es correcto: un bot sin tiempo registrado no aporta ahorro real.
#    Imputar con mediana sesgaría el cálculo del total de dinero salvado.
tm['TiempoManualHoras'] = pd.to_numeric(tm['TiempoManualHoras'], errors='coerce')
tm['TiempoManualHoras'] = tm['TiempoManualHoras'].fillna(0)
print(f'  [OK] TiempoManualHoras: nulos → 0  (registros: {len(tm)})')
print(f'       Distribución: min={tm["TiempoManualHoras"].min():.4f}  '
      f'mediana={tm["TiempoManualHoras"].median():.4f}  '
      f'max={tm["TiempoManualHoras"].max():.4f}')

# 3. FechaSalidaProduccion → datetime
tm['FechaSalidaProduccion'] = pd.to_datetime(
    tm['FechaSalidaProduccion'], format='%d/%m/%Y', errors='coerce'
)
print(f'  [OK] FechaSalidaProduccion: {tm["FechaSalidaProduccion"].isna().sum()} NaT conservados')

# 4. Desarrollador: nulos → 'Sin asignar'
tm['Desarrollador'] = tm['Desarrollador'].fillna('Sin asignar')

# 5. Tecnologia: estandarizar capitalización
TECH_MAP = {'uipath': 'UiPath', 'power automate': 'Power Automate',
            'irpa': 'IRPA', 'python': 'Python'}
tm['Tecnologia'] = (tm['Tecnologia'].str.strip().str.lower()
                    .map(lambda x: TECH_MAP.get(x, x.title()) if pd.notna(x) else x))

print(f'  Shape final TiemposManuales: {tm.shape}')
tm.head(3)

## 5 · Limpieza — RolesAreas

| Columna | Problema | Estrategia |
|---------|----------|------------|
| `Proyecto` | 1 fila vacía | Copiar desde `NombreBot` |
| `Rol_Impactado` | Espacios y comillas extra | Strip + `'No especificado'` |
| `ValorHoraProyecto` | 2 filas vacías | Mediana por grupo `Tipo_Impacto` |

> Imputar por grupo es más preciso que la mediana global: los valores hora
> difieren significativamente entre `Ahorrado` y `Salvado`.

In [ ]:
ra = df_roles.copy()
ra = ra.replace('NULL', np.nan)

# 1. Proyecto vacío → NombreBot
mask = ra['Proyecto'].isna() | (ra['Proyecto'].str.strip() == '')
ra.loc[mask, 'Proyecto'] = ra.loc[mask, 'NombreBot']
print(f'  [OK] Proyecto: {mask.sum()} filas imputadas desde NombreBot')

# 2. Limpiar Rol_Impactado
ra['Rol_Impactado'] = ra['Rol_Impactado'].str.strip().str.strip('"').fillna('No especificado')

# 3. ValorHoraProyecto numérico + mediana por grupo
ra['ValorHoraProyecto'] = pd.to_numeric(ra['ValorHoraProyecto'], errors='coerce')
ra['ValorHoraProyecto'] = ra.groupby('Tipo_Impacto')['ValorHoraProyecto'].transform(
    lambda g: g.fillna(g.median())
)
ra['ValorHoraProyecto'] = ra['ValorHoraProyecto'].fillna(ra['ValorHoraProyecto'].median())
print('  [OK] ValorHoraProyecto imputado por mediana de grupo')

print(f'  Shape final RolesAreas: {ra.shape}')
ra.head(3)

## 5 · Limpieza — RegistrosDPA

Dataset más grande (~70 MB). Operaciones aplicadas:
- Reemplazar `'NULL'` literal por `NaN`.
- Eliminar duplicados exactos.
- Parsear columnas de fecha automáticamente.
- Strip en columnas de texto.

> Los duplicados se eliminan **antes** de cualquier imputación para evitar
> que filas repetidas inflen artificialmente los estadísticos.

In [ ]:
dr = df_registros.copy()
dr = dr.replace('NULL', np.nan)

# 1. Duplicados
antes = len(dr)
dr = dr.drop_duplicates()
print(f'  [OK] Duplicados eliminados: {antes - len(dr):,}')

# 2. Columnas de fecha
cols_fecha = [c for c in dr.columns if 'fecha' in c.lower() or 'date' in c.lower()]
for col in cols_fecha:
    dr[col] = pd.to_datetime(dr[col], errors='coerce', dayfirst=True)
    print(f'  [OK] {col} → datetime ({dr[col].isna().sum():,} NaT)')

# 3. Strip en strings
for col in dr.select_dtypes(include=['object', 'str']).columns:
    dr[col] = dr[col].str.strip()

nulos_res = dr.isnull().sum()
nulos_res = nulos_res[nulos_res > 0]
print(f'\n  Nulos residuales:\n{nulos_res.to_string() if len(nulos_res) else "  Ninguno"}')
print(f'  Shape final RegistrosDPA: {dr.shape}')

## 6 · Normalización Numérica

**Min-Max Scaling** → rango [0, 1] sobre columnas numéricas clave.

| Técnica | Cuándo usarla |
|---------|---------------|
| **Min-Max** | Distribución asimétrica; modelos KNN, SVM, redes neuronales |
| **Z-Score** | Distribución normal; regresión lineal, PCA |

> Se elige Min-Max porque `TiempoManualHoras` y `ValorHoraProyecto` tienen
> distribuciones asimétricas con outliers claros.

In [ ]:
scaler = MinMaxScaler()

tm['TiempoManualHoras_norm'] = scaler.fit_transform(tm[['TiempoManualHoras']])
print('TiempoManualHoras — antes/después:')
print(tm[['TiempoManualHoras', 'TiempoManualHoras_norm']].describe().round(4))

ra['ValorHoraProyecto_norm'] = scaler.fit_transform(ra[['ValorHoraProyecto']])
print('\n  [OK] ValorHoraProyecto normalizado en RolesAreas')

## 7 · Exportación → Procesos_clean.db

Los datos limpios se escriben como nuevas tablas en `Procesos_clean.db`.
`Procesos.db` **no se modifica en ningún momento**.

Las columnas `datetime` se convierten a string antes de insertar en SQLite
(SQLite no tiene tipo nativo `DATE`).

In [ ]:
DATASETS = {
    'TiemposManuales_clean': tm,
    'RolesAreas_clean':      ra,
    'RegistrosDPA_clean':    dr,
}

with sqlite3.connect(DB_OUTPUT) as conn_w:
    for nombre, df_out in DATASETS.items():
        df_sqlite = df_out.copy()
        # Convertir datetime a string para compatibilidad SQLite
        for col in df_sqlite.select_dtypes(include='datetime64[ns]').columns:
            df_sqlite[col] = df_sqlite[col].dt.strftime('%Y-%m-%d')
        df_sqlite.to_sql(nombre, conn_w, if_exists='replace', index=False)
        print(f'  [OK] Tabla "{nombre}" → {df_out.shape[0]:,} filas × {df_out.shape[1]} cols')

size_clean = os.path.getsize(DB_OUTPUT) / 1_048_576
print(f'\n✅ Procesos_clean.db generado  ({size_clean:.1f} MB)')
print(f'   Ruta: {DB_OUTPUT}')
print(f'   Procesos.db original: intacto ✓')